# Idealized Synthetic Data

*Under development*

In [22]:
import numpy as np
from IPython.display import display  # so can run as script too

from melodies_monet import driver
from melodies_monet.tutorial import model, pt_sfc_obs

In [ ]:
an = driver.analysis()
an.control = "control_idealized.yaml"
an.read_control()
an

````{admonition} Note: This is the complete file that was loaded.
:class: dropdown

```{literalinclude} control_idealized.yaml
:caption:
:linenos:
```
````

## Generate data

### Model

In [ ]:
rs = np.random.RandomState(42)

control = an.control_dict


ds_mod = model(control, freq="3h")
ds_mod

In [ ]:
ds_mod.squeeze("z").A.plot(col="time", col_wrap=5, size=3);

In [5]:
ds_mod.to_netcdf(control['model']['idealized']['files'])

### Obs

In [ ]:
ds = pt_sfc_obs(control, model=ds_mod)
ds

In [7]:
ds.to_netcdf(control['obs']['test_obs']['filename'])

In [ ]:

def generate_swath_grid(
    lat,
    lon,
    alt,
    view_along=(-10, 10, 20),
    view_cross=(-5, 5, 10),
):
    """
    Generate a lat-lon grid for satellite swath data.
    
    Parameters:
    -----------
    lat : float
        Satellite latitude [deg]
    lon : float
        Satellite longitude [deg]
    alt : float
        Satellite altitude (above spherical Earth surface) [km].
        For example, ~ 35,786 km for geostationary orbit,
        200--1000 km for low Earth orbits (e.g. polar-orbiting satellites).
    view_along : tuple
        min and max viewing angles along track [deg], number of points
    view_cross : tuple
        min and max viewing angles across track [deg], number of points

    Returns:
    --------
    tuple
        (lats, lons) arrays of shape ``(n_along, n_cross)``
    """
    import numpy as np

    sat_lat_rad = np.deg2rad(lat)
    sat_lon_rad = np.deg2rad(lon)
    
    R_EARTH = 6371.0  # km
    
    # Satellite position in Earth-centered Cartesian coordinates
    r = R_EARTH + alt
    sat_x = r * np.cos(sat_lat_rad) * np.cos(sat_lon_rad)
    sat_y = r * np.cos(sat_lat_rad) * np.sin(sat_lon_rad)
    sat_z = r * np.sin(sat_lat_rad)
    sat_pos = np.array([sat_x, sat_y, sat_z])
    
    # Generate viewing angles
    # TODO: allow passing in arrays
    along_angles = np.deg2rad(lat + np.linspace(*view_along))
    cross_angles = np.deg2rad(lon + np.linspace(*view_cross))
    
    # Initialize output arrays
    lats = np.full((view_along[-1], view_cross[-1]), np.nan)
    lons = np.full((view_along[-1], view_cross[-1]), np.nan)
    
    # # Generate rotation matrices
    # # Rotation around z-axis (longitude)
    # Rz = np.array([[np.cos(sat_lon_rad), -np.sin(sat_lon_rad), 0],
    #                [np.sin(sat_lon_rad), np.cos(sat_lon_rad), 0],
    #                [0, 0, 1]])
    
    # # Rotation around y-axis (latitude)
    # Ry = np.array([[np.cos(sat_lat_rad), 0, np.sin(sat_lat_rad)],
    #                [0, 1, 0],
    #                [-np.sin(sat_lat_rad), 0, np.cos(sat_lat_rad)]])
    
    # # Combined rotation matrix
    # R = Ry @ Rz

    # Generate rotation matrices
    # Rotation around z-axis for longitude
    Rz = np.array([[np.cos(sat_lon_rad), np.sin(sat_lon_rad), 0],
                   [-np.sin(sat_lon_rad), np.cos(sat_lon_rad), 0],
                   [0, 0, 1]])
    
    # Rotation around y-axis for latitude
    Ry = np.array([[np.cos(sat_lat_rad), 0, -np.sin(sat_lat_rad)],
                   [0, 1, 0],
                   [np.sin(sat_lat_rad), 0, np.cos(sat_lat_rad)]])
    
    # Combined rotation matrix
    R = Rz @ Ry

    # Calculate grid points
    for i, along in enumerate(along_angles):
        for j, cross in enumerate(cross_angles):
            # Create viewing vector
            # view_vector = np.array([
            #     np.cos(along) * np.cos(cross),
            #     np.sin(cross),
            #     np.sin(along) * np.cos(cross)
            # ])
            view_vector = np.array([
                np.cos(along) * np.cos(cross),
                np.sin(cross),
                -np.sin(along) * np.cos(cross)
            ])
            
            # Rotate viewing vector to Earth frame
            view_vector = R @ view_vector
            
            # Find intersection with Earth surface
            a = np.dot(view_vector, view_vector)
            b = 2 * np.dot(sat_pos, view_vector)
            c = np.dot(sat_pos, sat_pos) - R_EARTH**2
            
            # Quadratic formula
            d = b**2 - 4*a*c
            if d < 0:
                continue  # No intersection with Earth
            
            # Use nearest intersection
            t1 = (-b - np.sqrt(d)) / (2*a)
            t2 = (-b + np.sqrt(d)) / (2*a)
            t = t1 if abs(t1) < abs(t2) else t2

            # Calculate intersection point
            point = sat_pos + t * view_vector
            
            # Convert to lat/lon
            lat = np.arcsin(point[2] / R_EARTH)
            lon = np.arctan2(point[1], point[0])
            
            lats[i, j] = np.rad2deg(lat)
            lons[i, j] = np.rad2deg(lon)
    
    return lats, lons

lat, lon = generate_swath_grid(0, -91, 35_786)  # TEMPO

# lat, lon = generate_swath_grid(0, 0, 705)
# lat, lon = generate_swath_grid(0, 0, 705, (-30, 30, 20), (-45, 45, 10))
lat, lon = generate_swath_grid(0, 45, 705) #, view_cross=(40, 50, 10))


import matplotlib.pyplot as plt

plt.scatter(lon, lat)

In [ ]:
import numpy as np
from pyproj import Geod

def calculate_swath_grid(
    lat,
    lon,
    alt,
    *,
    view_along=(-10, 10, 20),
    view_cross=(-5, 5, 10),
    look_angle=0,
    look_azimuth=0,
):
    """
    Generate a lat-lon grid for satellite swath data.
    
    Parameters:
    -----------
    lat : float
        Satellite latitude [deg]
    lon : float
        Satellite longitude [deg]
    alt : float
        Satellite altitude (above spherical Earth surface) [km].
        For example, ~ 35,786 km for geostationary orbit,
        200--1000 km for low Earth orbits (e.g. polar-orbiting satellites).
    view_along : tuple
        min and max viewing angles along track (y) [deg], number of points
    view_cross : tuple
        min and max viewing angles across track (x) [deg], number of points
    look_angle : float
        Angle from nadir [deg], e.g. 0 = straight down
    look_azimuth : float
        Azimuth angle of the look direction [deg], e.g. 0=N, 90=E
    """
    if look_angle != 0:
        raise NotImplementedError("nonzero look angles not really working")

    geod = Geod(ellps="WGS84")
    r_e_m = geod.a  # Earth radius (m)
    alt_m = alt * 1e3  # km -> m

    # Calculate maximum observable angle (theta_max)
    theta_max = np.arcsin(r_e_m / (r_e_m + alt_m))  # radians
    theta_max_deg = np.rad2deg(theta_max)

    # Create angular grid
    x_angles = np.linspace(*view_cross)
    y_angles = np.linspace(*view_along)
    xx, yy = np.meshgrid(x_angles, y_angles)

    # Calculate ground positions
    ny, nx = xx.shape
    lats, lons = [], []
    for i in range(ny):
        for j in range(nx):
            # Calculate combined angle from nadir
            theta_deg = np.sqrt(xx[i,j]**2 + yy[i,j]**2) + look_angle
            if theta_deg > theta_max_deg:
                # Point is beyond observable horizon
                lats.append(np.nan)
                lons.append(np.nan)
                continue

            # Convert to radians for calculations
            theta_rad = np.deg2rad(theta_deg)
            
            # Calculate effective azimuth
            effective_azimuth = look_azimuth + np.rad2deg(np.arctan2(xx[i,j], yy[i,j]))

            # # Calculate ground distance using flat Earth approximation
            # ground_distance = alt_m * np.arctan(theta_rad)
            
            # Calculate ground distance using spherical geometry
            sin_theta = np.sin(theta_rad)
            sin_gamma = (r_e_m + alt_m) / r_e_m * sin_theta
            gamma = np.arcsin(sin_gamma)
            ground_distance = r_e_m * gamma  # Along Earth's surface

            # Calculate new coordinates
            lon_new, lat_new, _ = geod.fwd(
                lon, lat,
                effective_azimuth,
                ground_distance,
                radians=False,
            )
            lats.append(lat_new)
            lons.append(lon_new)

    return np.array(lats), np.array(lons)


# Generate swath grid (TEMPO-like)
lats, lons = calculate_swath_grid(
    0, -91, 35_786,
    view_cross=(-8, 8, 30),
    view_along=(-8, 8, 20),
    look_angle=0,
    look_azimuth=1.23,
)

# Print first few points
print(f"{len(lats)} points {np.isnan(lons).sum()/len(lons):.2%} null")
print("First 5 swath points:")
for i in range(5):
    if np.isnan(lons[i]):
        continue
    print(f"Point {i+1}: {lats[i]:.4f}°, {lons[i]:.4f}°")

import cartopy.crs as ccrs
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(lons, lats)

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Geostationary(central_longitude=sat_lon))
ax.coastlines()
ax.scatter(lons, lats, transform=ccrs.PlateCarree())

# Also plot opposite side of Earth
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Geostationary(central_longitude=(sat_lon + 180) % 360))
ax.coastlines()
ax.scatter(lons, lats, transform=ccrs.PlateCarree())



## Load

In [ ]:
an.open_models()

In [ ]:
an.models['idealized'].obj

In [11]:
an.open_obs()

In [ ]:
an.obs['test_obs'].obj

## Pair

In [ ]:
%%time

an.pair_data()

In [ ]:
an.paired

In [ ]:
an.paired['test_obs_idealized'].obj

In [ ]:
an.paired['test_obs_idealized'].obj.dims

## Plot

In [ ]:
%%time

an.plotting()

## Save/load paired data -- netCDF

And compare to the original pair object.

In [ ]:
from copy import deepcopy

p0 = deepcopy(an.paired['test_obs_idealized'].obj)

an.save_analysis()
an.read_analysis()
p1 = deepcopy(an.paired['test_obs_idealized'].obj)
p1.close()

display(p0)
display(p1)
assert p1 is not p0 and p1.equals(p0)

## Save/load paired data -- Python object

In [ ]:
print(an.save)
an.save["paired"]["method"] = "pkl"
del an.save["paired"]["prefix"]
# We could leave `prefix` since unused, but we need to set `output_name`
an.save["paired"]["output_name"] = "asdf.joblib"
print("->", an.save)

print()
print(an.read)
an.read["paired"]["method"] = "pkl"
an.read["paired"]["filenames"] = "asdf.joblib"
print("->", an.read)

print()
an.save_analysis()
an.read_analysis()
p2 = deepcopy(an.paired['test_obs_idealized'].obj)
p2.close()

# display(p0)
display(p2)
assert p2 is not p0 and p2.equals(p0)